In [ ]:
!pip install optuna

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

from torch.utils.data import Dataset, DataLoader


In [ ]:
torch.manual_seed(42)

In [ ]:
#I loaded this with AI
import torchvision
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

# Download and load the training data
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)

# Download and load the test data
test_data = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)

cpu


In [ ]:
# Extract images and labels from the PyTorch dataset
# Flatten the 28x28 images into 784 features
X_raw = train_data.data.numpy().reshape(-1, 28*28)
y_raw = train_data.targets.numpy()
X_test_raw = test_data.data.numpy().reshape(-1, 28*28)
y_test_raw = test_data.targets.numpy()

data = pd.DataFrame(X_test_raw)
data.insert(0, 'label', y_test_raw)

# Create a DataFrame
df = pd.DataFrame(X_raw)

# Insert the label as the first column to match your previous setup
df.insert(0, 'label', y_raw)
df


,label,0,1,2,3,4,5,6,7,8,...,774,775,776,777,778,779,780,781,782,783
0,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,...,119,114,130,76,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,33,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,0,0,0,0,0,0,0,0,0,0,...,66,54,50,5,0,1,0,0,0,0


In [ ]:
X_train = df.iloc[:,1:].values/255.0
y_train = df.iloc[:,0].values

X_test = data.iloc[:,1:].values/255.0
y_test = data.iloc[:,0].values

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, feature, label):
    self.feature = torch.tensor(feature, dtype = torch.float32)
    self.label = torch.tensor(label, dtype = torch.long)
  def __len__(self):
    return len(self.feature)

  def __getitem__(self,idx):
    return self.feature[idx], self.label[idx]


In [ ]:
train_data = CustomDataset(X_train,y_train)
test_data = CustomDataset(X_test,y_test)

In [ ]:
class MyNN(nn.Module):
  def __init__(self, input_feature, kernel_conv_size, kernel_max_size,stride,dropout_rate):

    super().__init__()
    self.model = nn.Sequential(
        nn.Conv2d(input_feature,32, kernel_size = kernel_conv_size, padding= 'same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size = kernel_max_size, stride = stride),

        nn.Conv2d(32,64,kernel_size = kernel_conv_size ,stride=stride),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size = kernel_max_size),
        nn.AdaptiveAvgPool2d((7, 7)) # Added this to ensure output is 7x7
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),

        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p = dropout_rate),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),

        nn.Linear(64,10)
    )
  def forward(self, x):
    x = self.model(x)
    x = self.classifier(x)


In [ ]:
#My objective function
def objective(trial):

  #hyperparameter

  kernel_conv_size = trial.suggest_categorical('kernel_size',[3,5,7])
  kernel_max_size = trial.suggest_categorical('kernel_max_size',[3,5,7])
  dropout_rate = trial.suggest_float('dropout_rate', 0.1,0.5,step = 0.1)
  neuron_per_layer = 0
  batch_size = trial.suggest_categorical('Batch size', [32,64,128,256])
  weight_decay = trial.suggest_float('weight_Decay', 1e-5,1e-3, log = True)
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log= True)
  stride = trial.suggest_int('stride',1,3)
  optimizer = trial.suggest_categorical('optimizer',['Adam','SGD', 'RMSprop','AdamW','AdaGrad'])
  epochs = trial.suggest_int('epoch',50,150,step = 25)

  #model init
  model = MyNN(1, kernel_conv_size, kernel_max_size, stride, dropout_rate).to(device)
  criterian = nn.CrossEntropyLoss()

  #for the optimizer

  if optimizer == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  elif optimizer == 'SGD':
    optimizer = optim.SGD(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  elif optimizer =='RMSprop':
    optimizer = optim.RMSprop(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  elif optimizer == 'AdamW':
    optimizer = optim.AdamW(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
  else:
    optimizer = optim.Adagrad(model.parameters(), lr = learning_rate, weight_decay = weight_decay)



  #train test Loader
  train_loader = DataLoader(train_data, batch_size = batch_size, shuffle = True)
  test_loader = DataLoader(test_data, batch_size = batch_size, shuffle = False)

  # for training the mdoel
  for epoch in range(epochs):

    for batch_feature, batch_label in train_loader:
      batch_feature, batch_label = batch_feature.to(device), batch_label.to(device)
      batch_feature = batch_feature.view(-1, 1, 28, 28) # Reshape to 4D

      optimizer.zero_grad()
      #forward
      output = model(batch_feature)

      #loss calculation
      loss = criterian(output,b atch_label)
      # backpropagation
      loss.backward()
      #optimization
      optimizer.step()



  #model evaluation
  model.eval()

  # Evaluation
  total = 0
  correct = 0

  with torch.no_grad():
    for batch_feature, batch_label in test_loader:
      batch_feature, batch_label = batch_feature.to(device), batch_label.to(device)
      batch_feature = batch_feature.view(-1, 1, 28, 28) # Reshape to 4D

      output = model(batch_feature)

      _, predicted = torch.max(output, 1)
      total = total + batch_label.shape[0]
      correct = correct + (predicted == batch_label).sum().item()
  accuracy = correct/total

  return accuracy


In [ ]:
#study for optuna
study = optuna.create_study(direction = 'maximize')

[I 2026-04-27 05:29:39,683] A new study created in memory with name: no-name-01b60e6c-8ad9-4447-ab90-bee9a2f5ba2b


In [ ]:
class MyNN(nn.Module):
  def __init__(self, input_feature, kernel_conv_size, kernel_max_size,stride,dropout_rate):

    super().__init__()
    self.model = nn.Sequential(
        nn.Conv2d(input_feature,32, kernel_size = kernel_conv_size, padding= 'same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size = kernel_max_size, stride = stride),

        nn.Conv2d(32,64,kernel_size = kernel_conv_size ,stride=stride),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size = kernel_max_size),
        nn.AdaptiveAvgPool2d((7, 7)) # Added this to ensure output is 7x7
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),

        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p = dropout_rate),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),

        nn.Linear(64,10)
    )
  def forward(self, x):
    x = self.model(x)
    x = self.classifier(x)
    return x

study.optimize(objective, n_trials = 10)


[I 2026-04-27 05:43:14,034] Trial 1 finished with value: 0.8608 and parameters: {'kernel_size': 7, 'kernel_max_size': 3, 'dropout_rate': 0.4, 'Batch size': 64, 'weight_Decay': 3.476728303920642e-05, 'learning_rate': 0.006811625937248662, 'stride': 2, 'optimizer': 'Adam', 'epoch': 75}. Best is trial 1 with value: 0.8608.
[I 2026-04-27 06:09:47,174] Trial 2 finished with value: 0.9046 and parameters: {'kernel_size': 7, 'kernel_max_size': 5, 'dropout_rate': 0.5, 'Batch size': 32, 'weight_Decay': 0.00012205123738999114, 'learning_rate': 0.0014110468772920233, 'stride': 1, 'optimizer': 'Adam', 'epoch': 50}. Best is trial 2 with value: 0.9046.
[I 2026-04-27 06:20:05,209] Trial 3 finished with value: 0.8919 and parameters: {'kernel_size': 3, 'kernel_max_size': 3, 'dropout_rate': 0.2, 'Batch size': 32, 'weight_Decay': 0.0006103261228470136, 'learning_rate': 0.00023548060276085708, 'stride': 3, 'optimizer': 'SGD', 'epoch': 75}. Best is trial 2 with value: 0.9046.
[W 2026-04-27 06:20:05,225] Tri

RuntimeError: Given input size: (64x2x2). Calculated output size: (64x0x0). Output size is too small